# Apache Spark RDD Transformations — Complete Detailed Notebook


## 1. Spark Setup

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[*]")
    .appName("Complete RDD Transformations")
    .getOrCreate()
)

sc = spark.sparkContext

print("Spark Version:", spark.version)
print("Application Name:", sc.appName)
print("Default Parallelism:", sc.defaultParallelism)

# 2. RDD Transformation Fundamentals

An **RDD transformation** creates a new RDD from an existing RDD.

Transformations are **lazy**, which means Spark does not immediately execute them.

Example:

```python
rdd2 = rdd1.map(lambda x: x * 2)
```

Spark only records this operation in the lineage.

Execution starts when an action is called:

```python
rdd2.collect()
```

## Narrow transformation

An output partition depends on one input partition.

Usually no shuffle.

Examples:

- `map()`
- `filter()`
- `flatMap()`
- `mapPartitions()`

## Wide transformation

An output partition depends on multiple input partitions.

Usually requires shuffle.

Examples:

- `reduceByKey()`
- `groupByKey()`
- `join()`
- `distinct()`
- `sortByKey()`

# 3. `map()`

## Definition

Applies a function to every element and returns exactly one output element for each input element.

## Why do we use it?

Use it when each input record must be converted into one transformed record.

## Transformation classification

- **Type:** Narrow
- **Shuffle:** No

## Conceptual flow

```text
[1, 2, 3] → map(x * x) → [1, 4, 9]
```

## Syntax

```python
new_rdd = rdd.map(function)
```

## PySpark example

In [ ]:
numbers = sc.parallelize([1, 2, 3, 4, 5])
squares = numbers.map(lambda x: x * x)
print(squares.collect())

## Expected output

```text
[1, 4, 9, 16, 25]
```

## Real-world use case

Convert raw CSV strings into structured tuples, normalize names, calculate tax or salary values.

## Performance notes

Very efficient because each partition is processed independently and no shuffle is required.

## Practice questions

1. Convert [10, 20, 30] into [20, 40, 60].
2. Convert employee names to uppercase.
3. Convert temperatures from Celsius to Fahrenheit.

# 4. `flatMap()`

## Definition

Applies a function to each element and flattens the returned collections into one RDD.

## Why do we use it?

Use it when one input record can create zero, one, or many output records.

## Transformation classification

- **Type:** Narrow
- **Shuffle:** No

## Conceptual flow

```text
['Spark is fast'] → split → ['Spark', 'is', 'fast']
```

## Syntax

```python
new_rdd = rdd.flatMap(function)
```

## PySpark example

In [ ]:
sentences = sc.parallelize([
    "Spark is fast",
    "Spark is powerful"
])

words = sentences.flatMap(lambda line: line.split())
print(words.collect())

## Expected output

```text
['Spark', 'is', 'fast', 'Spark', 'is', 'powerful']
```

## Real-world use case

Split log lines, tokenize text, extract multiple products from one order record.

## Performance notes

No shuffle by itself. Output size can become much larger than input, so memory usage must be considered.

## Practice questions

1. Split ['AWS Glue', 'Spark RDD'] into individual words.
2. Flatten [[1, 2], [3, 4], [5]].
3. Extract all hashtags from a list of social media posts.

# 5. `filter()`

## Definition

Keeps only records for which the condition returns True.

## Why do we use it?

Use it to remove unwanted, invalid, failed, or irrelevant records.

## Transformation classification

- **Type:** Narrow
- **Shuffle:** No

## Conceptual flow

```text
[1, 2, 3, 4] → keep even → [2, 4]
```

## Syntax

```python
new_rdd = rdd.filter(condition)
```

## PySpark example

In [ ]:
numbers = sc.parallelize(range(1, 11))
even_numbers = numbers.filter(lambda x: x % 2 == 0)
print(even_numbers.collect())

## Expected output

```text
[2, 4, 6, 8, 10]
```

## Real-world use case

Filter failed transactions, invalid rows, high-value customers, or ERROR logs.

## Performance notes

No shuffle. Highly selective filters can reduce later processing and shuffle cost.

## Practice questions

1. Keep numbers greater than 50.
2. Filter employees whose salary is at least 70000.
3. Keep log lines starting with ERROR.

# 6. `mapPartitions()`

## Definition

Runs one function once per partition instead of once per record.

## Why do we use it?

Use it when creating an expensive resource once per partition is better than creating it for every record.

## Transformation classification

- **Type:** Narrow
- **Shuffle:** No

## Conceptual flow

```text
Partition [1,2,3] → one function call → result
```

## Syntax

```python
new_rdd = rdd.mapPartitions(function)
```

## PySpark example

In [ ]:
numbers = sc.parallelize(range(1, 11), 3)

def partition_sum(iterator):
    values = list(iterator)
    return iter([sum(values)])

result = numbers.mapPartitions(partition_sum)
print(result.collect())

## Expected output

```text
One sum value per partition
```

## Real-world use case

Open one database connection per partition, create one API client per partition, load one model per partition.

## Performance notes

Can be faster than map(), but loading an entire partition into memory can cause problems.

## Practice questions

1. Count records in every partition.
2. Calculate maximum value in every partition.
3. Return the first element from each partition.

# 7. `mapPartitionsWithIndex()`

## Definition

Processes every partition and also provides its partition index.

## Why do we use it?

Use it when partition number is required for debugging or custom processing.

## Transformation classification

- **Type:** Narrow
- **Shuffle:** No

## Conceptual flow

```text
Partition 0 → values; Partition 1 → values
```

## Syntax

```python
rdd.mapPartitionsWithIndex(function)
```

## PySpark example

In [ ]:
numbers = sc.parallelize(range(1, 11), 3)

def inspect(index, iterator):
    return iter([(index, list(iterator))])

result = numbers.mapPartitionsWithIndex(inspect)
print(result.collect())

## Expected output

```text
[(0, [...]), (1, [...]), (2, [...])]
```

## Real-world use case

Inspect data skew, label partition output, debug distribution.

## Performance notes

No shuffle. Mainly useful for partition awareness.

## Practice questions

1. Print partition number and record count.
2. Return partition number with partition sum.
3. Identify empty partitions.

# 8. `glom()`

## Definition

Converts each partition into a Python list.

## Why do we use it?

Use it to inspect how records are distributed among partitions.

## Transformation classification

- **Type:** Narrow
- **Shuffle:** No

## Conceptual flow

```text
Partitions → list per partition
```

## Syntax

```python
rdd.glom()
```

## PySpark example

In [ ]:
numbers = sc.parallelize(range(1, 11), 3)
print(numbers.glom().collect())

## Expected output

```text
A list containing one list per partition
```

## Real-world use case

Debug partition distribution and explain Spark partitioning.

## Performance notes

Use only for small datasets because collect() brings partition contents to the driver.

## Practice questions

1. Create 4 partitions and inspect them.
2. Compare glom() before and after repartition().
3. Find which partition has the most records.

# 9. `union()`

## Definition

Combines two RDDs and preserves duplicates.

## Why do we use it?

Use it to append datasets having compatible record structures.

## Transformation classification

- **Type:** Narrow in many cases
- **Shuffle:** Usually No

## Conceptual flow

```text
[1,2] + [2,3] → [1,2,2,3]
```

## Syntax

```python
rdd1.union(rdd2)
```

## PySpark example

In [ ]:
rdd1 = sc.parallelize([1, 2, 3])
rdd2 = sc.parallelize([3, 4, 5])
result = rdd1.union(rdd2)
print(result.collect())

## Expected output

```text
[1, 2, 3, 3, 4, 5]
```

## Real-world use case

Combine daily files, historical and incremental data, or logs from multiple sources.

## Performance notes

Duplicates remain. Add distinct() only when uniqueness is required.

## Practice questions

1. Combine two employee RDDs.
2. Combine January and February sales.
3. Union two RDDs and then remove duplicates.

# 10. `intersection()`

## Definition

Returns common elements from two RDDs and removes duplicates.

## Why do we use it?

Use it to identify records present in both datasets.

## Transformation classification

- **Type:** Wide
- **Shuffle:** Yes

## Conceptual flow

```text
[1,2,3] ∩ [2,3,4] → [2,3]
```

## Syntax

```python
rdd1.intersection(rdd2)
```

## PySpark example

In [ ]:
rdd1 = sc.parallelize([1, 2, 3, 4])
rdd2 = sc.parallelize([3, 4, 5, 6])
print(sorted(rdd1.intersection(rdd2).collect()))

## Expected output

```text
[3, 4]
```

## Real-world use case

Find customers present in two campaigns or common products across systems.

## Performance notes

Requires shuffle and duplicate removal.

## Practice questions

1. Find common students between two classes.
2. Find common product IDs.
3. Find users active in both months.

# 11. `subtract()`

## Definition

Returns elements from the first RDD that do not appear in the second RDD.

## Why do we use it?

Use it for exclusion or difference analysis.

## Transformation classification

- **Type:** Wide
- **Shuffle:** Yes

## Conceptual flow

```text
[1,2,3,4] - [2,4] → [1,3]
```

## Syntax

```python
rdd1.subtract(rdd2)
```

## PySpark example

In [ ]:
rdd1 = sc.parallelize([1, 2, 3, 4, 5])
rdd2 = sc.parallelize([2, 4])
print(sorted(rdd1.subtract(rdd2).collect()))

## Expected output

```text
[1, 3, 5]
```

## Real-world use case

Find unprocessed records, missing customers, or inventory not sold.

## Performance notes

Requires shuffle.

## Practice questions

1. Find students who did not attend.
2. Find files not yet processed.
3. Find inactive customers.

# 12. `distinct()`

## Definition

Removes duplicate records from an RDD.

## Why do we use it?

Use it when unique records are required.

## Transformation classification

- **Type:** Wide
- **Shuffle:** Yes

## Conceptual flow

```text
[1,2,2,3] → [1,2,3]
```

## Syntax

```python
rdd.distinct()
```

## PySpark example

In [ ]:
numbers = sc.parallelize([1, 2, 2, 3, 3, 3, 4])
print(sorted(numbers.distinct().collect()))

## Expected output

```text
[1, 2, 3, 4]
```

## Real-world use case

Remove duplicate customer IDs, emails, file names, or transactions.

## Performance notes

Expensive because equal values must be moved together through shuffle.

## Practice questions

1. Remove duplicate names.
2. Remove duplicate email addresses.
3. Count unique product IDs.

# 13. `cartesian()`

## Definition

Returns every possible pair between two RDDs.

## Why do we use it?

Use it only when all combinations are genuinely needed.

## Transformation classification

- **Type:** Wide/Expensive
- **Shuffle:** High data expansion

## Conceptual flow

```text
[1,2] × [A,B] → (1,A),(1,B),(2,A),(2,B)
```

## Syntax

```python
rdd1.cartesian(rdd2)
```

## PySpark example

In [ ]:
numbers = sc.parallelize([1, 2])
letters = sc.parallelize(["A", "B"])
print(numbers.cartesian(letters).collect())

## Expected output

```text
[(1,'A'), (1,'B'), (2,'A'), (2,'B')]
```

## Real-world use case

Generate combinations for small lookup sets, test cases, or recommendation candidates.

## Performance notes

If inputs contain m and n records, output has m × n records. Avoid on large datasets.

## Practice questions

1. Generate all shirt-size combinations.
2. Create all student-subject pairs.
3. Calculate output count for 1000 × 500 records.

# 14. `sample()`

## Definition

Returns a random sample from an RDD.

## Why do we use it?

Use it for testing, analysis, model development, or approximate inspection.

## Transformation classification

- **Type:** Narrow
- **Shuffle:** No

## Conceptual flow

```text
100 records → sample 10% → approximately 10 records
```

## Syntax

```python
rdd.sample(withReplacement, fraction, seed)
```

## PySpark example

In [ ]:
numbers = sc.parallelize(range(1, 101))
sampled = numbers.sample(False, 0.10, seed=42)
print(sampled.collect())

## Expected output

```text
Approximately 10% of records
```

## Real-world use case

Create development samples from large production datasets.

## Performance notes

The returned count is approximate, not guaranteed.

## Practice questions

1. Sample 20% without replacement.
2. Sample with replacement.
3. Run with and without a fixed seed.

# 15. `keyBy()`

## Definition

Creates a Pair RDD by generating a key for every element.

## Why do we use it?

Use it when ordinary records must become key-value records.

## Transformation classification

- **Type:** Narrow
- **Shuffle:** No

## Conceptual flow

```text
apple → ('a','apple')
```

## Syntax

```python
rdd.keyBy(function)
```

## PySpark example

In [ ]:
words = sc.parallelize(["apple", "banana", "mango"])
print(words.keyBy(lambda word: word[0]).collect())

## Expected output

```text
[('a','apple'), ('b','banana'), ('m','mango')]
```

## Real-world use case

Create keys for joins, aggregations, and partitioning.

## Performance notes

No shuffle until a key-based wide operation is applied.

## Practice questions

1. Key employees by department.
2. Key words by length.
3. Key numbers as even or odd.

# 16. `keys() and values()`

## Definition

keys() returns all keys; values() returns all values from a Pair RDD.

## Why do we use it?

Use them to extract one side of key-value records.

## Transformation classification

- **Type:** Narrow
- **Shuffle:** No

## Conceptual flow

```text
[(1,A),(2,B)] → keys [1,2], values [A,B]
```

## Syntax

```python
pair_rdd.keys(); pair_rdd.values()
```

## PySpark example

In [ ]:
employees = sc.parallelize([(101, "Anuj"), (102, "Rahul")])
print("Keys:", employees.keys().collect())
print("Values:", employees.values().collect())

## Expected output

```text
Keys [101,102], Values ['Anuj','Rahul']
```

## Real-world use case

Extract IDs, names, categories, or measures.

## Performance notes

No shuffle.

## Practice questions

1. Extract department names.
2. Extract salary values.
3. Count unique keys.

# 17. `mapValues()`

## Definition

Transforms only values while preserving keys.

## Why do we use it?

Use it when the key must remain unchanged.

## Transformation classification

- **Type:** Narrow
- **Shuffle:** No

## Conceptual flow

```text
(A,10) → (A,20)
```

## Syntax

```python
pair_rdd.mapValues(function)
```

## PySpark example

In [ ]:
scores = sc.parallelize([("Anuj", 80), ("Rahul", 70)])
print(scores.mapValues(lambda score: score + 5).collect())

## Expected output

```text
[('Anuj',85), ('Rahul',75)]
```

## Real-world use case

Apply tax, normalize metrics, convert currencies, or transform nested values.

## Performance notes

Can preserve partitioner information, unlike a general map().

## Practice questions

1. Increase salary by 10%.
2. Convert marks to percentages.
3. Convert values from rupees to dollars.

# 18. `flatMapValues()`

## Definition

Creates multiple values for each key while keeping the original key.

## Why do we use it?

Use it when one value contains multiple sub-values.

## Transformation classification

- **Type:** Narrow
- **Shuffle:** No

## Conceptual flow

```text
(Student, 'Spark AWS') → (Student,Spark),(Student,AWS)
```

## Syntax

```python
pair_rdd.flatMapValues(function)
```

## PySpark example

In [ ]:
courses = sc.parallelize([
    ("Student1", "Spark Python"),
    ("Student2", "AWS SQL")
])
print(courses.flatMapValues(lambda value: value.split()).collect())

## Expected output

```text
One record per key and split value
```

## Real-world use case

Expand product lists, course lists, tags, or categories.

## Performance notes

No shuffle, but output size can increase significantly.

## Practice questions

1. Split customer product lists.
2. Expand employee skills.
3. Expand comma-separated tags.

# 19. `groupBy()`

## Definition

Groups ordinary RDD elements using a key function.

## Why do we use it?

Use it when you need all original values organized under a generated group key.

## Transformation classification

- **Type:** Wide
- **Shuffle:** Yes

## Conceptual flow

```text
[1,2,3,4] → even:[2,4], odd:[1,3]
```

## Syntax

```python
rdd.groupBy(function)
```

## PySpark example

In [ ]:
numbers = sc.parallelize([1, 2, 3, 4, 5, 6])
grouped = numbers.groupBy(lambda x: "even" if x % 2 == 0 else "odd")
print(grouped.mapValues(list).collect())

## Expected output

```text
Groups for even and odd numbers
```

## Real-world use case

Group customers by age band or records by category.

## Performance notes

All values are shuffled and held in iterables. Avoid for simple aggregation.

## Practice questions

1. Group words by first letter.
2. Group salaries by salary range.
3. Group numbers by remainder when divided by 3.

# 20. `groupByKey()`

## Definition

Groups all values belonging to the same key.

## Why do we use it?

Use it only when all original values are required for each key.

## Transformation classification

- **Type:** Wide
- **Shuffle:** Yes

## Conceptual flow

```text
(A,1),(A,2) → (A,[1,2])
```

## Syntax

```python
pair_rdd.groupByKey()
```

## PySpark example

In [ ]:
data = sc.parallelize([
    ("HR", 50000),
    ("IT", 70000),
    ("HR", 60000)
])
print(data.groupByKey().mapValues(list).collect())

## Expected output

```text
[('HR',[50000,60000]), ('IT',[70000])]
```

## Real-world use case

Collect complete event lists or order lists per customer.

## Performance notes

Expensive. All values are shuffled. Prefer reduceByKey() for sum, min, max, and count.

## Practice questions

1. Group marks by student.
2. Group orders by customer.
3. Explain why groupByKey is inefficient for word count.

# 21. `reduceByKey()`

## Definition

Aggregates values for each key using an associative and commutative function.

## Why do we use it?

Use it for sums, counts, minimums, maximums, and similar aggregations.

## Transformation classification

- **Type:** Wide
- **Shuffle:** Yes, with map-side combine

## Conceptual flow

```text
Local reduce → shuffle partial totals → final reduce
```

## Syntax

```python
pair_rdd.reduceByKey(function)
```

## PySpark example

In [ ]:
sales = sc.parallelize([
    ("North", 100),
    ("South", 200),
    ("North", 300)
])
print(sales.reduceByKey(lambda a, b: a + b).collect())

## Expected output

```text
[('North',400), ('South',200)]
```

## Real-world use case

Word count, department totals, transaction counts, sales totals.

## Performance notes

More efficient than groupByKey because local aggregation reduces shuffled data.

## Practice questions

1. Calculate department salary totals.
2. Find maximum marks by student.
3. Count occurrences of each word.

# 22. `foldByKey()`

## Definition

Aggregates values per key starting with a zero value.

## Why do we use it?

Use it when aggregation requires an explicit identity value.

## Transformation classification

- **Type:** Wide
- **Shuffle:** Yes

## Conceptual flow

```text
Zero + partition values → partial result → final result
```

## Syntax

```python
pair_rdd.foldByKey(zero)(function)
```

## PySpark example

In [ ]:
sales = sc.parallelize([
    ("A", 10),
    ("A", 20),
    ("B", 30)
])
print(sales.foldByKey(0)(lambda a, b: a + b).collect())

## Expected output

```text
[('A',30), ('B',30)]
```

## Real-world use case

Totals and counters where a safe zero value exists.

## Performance notes

Zero value may be applied multiple times, so it must be an identity value.

## Practice questions

1. Calculate product quantity totals.
2. Concatenate strings carefully.
3. Explain why 10 is not a safe zero value for summation.

# 23. `aggregateByKey()`

## Definition

Uses one function to combine values inside a partition and another function to combine partition results.

## Why do we use it?

Use it when the accumulator type differs from the original value type.

## Transformation classification

- **Type:** Wide
- **Shuffle:** Yes, with local aggregation

## Conceptual flow

```text
Values → (sum,count) locally → merge partial (sum,count)
```

## Syntax

```python
rdd.aggregateByKey(zero)(seqFunc, combFunc)
```

## PySpark example

In [ ]:
scores = sc.parallelize([
    ("A", 10),
    ("A", 20),
    ("B", 30),
    ("A", 40),
    ("B", 50)
], 2)

sum_count = scores.aggregateByKey((0, 0))(
    lambda acc, value: (acc[0] + value, acc[1] + 1),
    lambda left, right: (left[0] + right[0], left[1] + right[1])
)

averages = sum_count.mapValues(lambda x: x[0] / x[1])

print(sum_count.collect())
print(averages.collect())

## Expected output

```text
Sum-count tuples and average by key
```

## Real-world use case

Average salary, average marks, sum and count, custom statistics.

## Performance notes

Efficient because local aggregation happens before shuffle.

## Practice questions

1. Calculate average salary by department.
2. Calculate min, max, and count per key.
3. Store sum and number of transactions per customer.

# 24. `combineByKey()`

## Definition

The most flexible Pair RDD aggregation. It creates and merges custom combiners.

## Why do we use it?

Use it for complex aggregation logic.

## Transformation classification

- **Type:** Wide
- **Shuffle:** Yes, with local aggregation

## Conceptual flow

```text
First value → combiner → merge values → merge partition combiners
```

## Syntax

```python
rdd.combineByKey(createCombiner, mergeValue, mergeCombiners)
```

## PySpark example

In [ ]:
scores = sc.parallelize([
    ("A", 10),
    ("A", 20),
    ("B", 30),
    ("A", 40)
], 2)

combined = scores.combineByKey(
    lambda value: (value, 1),
    lambda acc, value: (acc[0] + value, acc[1] + 1),
    lambda left, right: (left[0] + right[0], left[1] + right[1])
)

averages = combined.mapValues(lambda x: x[0] / x[1])

print(combined.collect())
print(averages.collect())

## Expected output

```text
Custom accumulator and average per key
```

## Real-world use case

Complex statistics, custom collections, averages, top-N per key.

## Performance notes

Powerful and efficient, but more complex to understand and maintain.

## Practice questions

1. Calculate average marks using combineByKey.
2. Collect unique values per key using a set.
3. Track sum, count, minimum, and maximum per key.

# 25. `join()`

## Definition

Performs an inner join between two Pair RDDs.

## Why do we use it?

Use it when only matching keys from both datasets are required.

## Transformation classification

- **Type:** Wide
- **Shuffle:** Yes

## Conceptual flow

```text
(1,Anuj) + (1,IT) → (1,(Anuj,IT))
```

## Syntax

```python
rdd1.join(rdd2)
```

## PySpark example

In [ ]:
employees = sc.parallelize([
    (1, "Anuj"),
    (2, "Rahul"),
    (3, "Priya")
])

departments = sc.parallelize([
    (1, "IT"),
    (2, "HR"),
    (4, "Finance")
])

print(employees.join(departments).collect())

## Expected output

```text
Only keys 1 and 2
```

## Real-world use case

Join customers with orders, employees with departments, products with prices.

## Performance notes

Shuffle can be expensive. Partition both RDDs consistently when joins are repeated.

## Practice questions

1. Join students with marks.
2. Join products with categories.
3. Explain what happens to unmatched keys.

# 26. `leftOuterJoin()`

## Definition

Returns all records from the left RDD and matching values from the right RDD.

## Why do we use it?

Use it when every left-side record must be preserved.

## Transformation classification

- **Type:** Wide
- **Shuffle:** Yes

## Conceptual flow

```text
Missing right value → None
```

## Syntax

```python
rdd1.leftOuterJoin(rdd2)
```

## PySpark example

In [ ]:
left = sc.parallelize([(1, "A"), (2, "B"), (3, "C")])
right = sc.parallelize([(1, 100), (2, 200)])
print(left.leftOuterJoin(right).collect())

## Expected output

```text
Key 3 contains None for the right-side value
```

## Real-world use case

Keep all customers even if some have no orders.

## Performance notes

Requires shuffle.

## Practice questions

1. Find employees without departments.
2. Keep all products even without sales.
3. Replace None with 0 using mapValues().

# 27. `rightOuterJoin()`

## Definition

Returns all records from the right RDD and matching values from the left RDD.

## Why do we use it?

Use it when every right-side record must be preserved.

## Transformation classification

- **Type:** Wide
- **Shuffle:** Yes

## Conceptual flow

```text
Missing left value → None
```

## Syntax

```python
rdd1.rightOuterJoin(rdd2)
```

## PySpark example

In [ ]:
left = sc.parallelize([(1, "A"), (2, "B")])
right = sc.parallelize([(1, 100), (2, 200), (3, 300)])
print(left.rightOuterJoin(right).collect())

## Expected output

```text
Key 3 contains None for the left-side value
```

## Real-world use case

Keep all reference records even if transactional data is missing.

## Performance notes

Requires shuffle.

## Practice questions

1. Keep all departments even without employees.
2. Keep all products in the master dataset.
3. Identify records missing from the left RDD.

# 28. `fullOuterJoin()`

## Definition

Returns every key from both RDDs.

## Why do we use it?

Use it for complete reconciliation.

## Transformation classification

- **Type:** Wide
- **Shuffle:** Yes

## Conceptual flow

```text
All keys retained; missing values become None
```

## Syntax

```python
rdd1.fullOuterJoin(rdd2)
```

## PySpark example

In [ ]:
left = sc.parallelize([(1, "A"), (2, "B")])
right = sc.parallelize([(2, 200), (3, 300)])
print(left.fullOuterJoin(right).collect())

## Expected output

```text
Keys 1, 2, and 3
```

## Real-world use case

Compare source and target systems and find missing records on either side.

## Performance notes

Requires shuffle and may create large output.

## Practice questions

1. Reconcile customer IDs.
2. Compare two inventory systems.
3. Classify matched, left-only, and right-only records.

# 29. `cogroup()`

## Definition

Groups values from multiple Pair RDDs by key into separate iterables.

## Why do we use it?

Use it when you need complete grouped values from each RDD rather than direct joined pairs.

## Transformation classification

- **Type:** Wide
- **Shuffle:** Yes

## Conceptual flow

```text
Key → (values from RDD1, values from RDD2)
```

## Syntax

```python
rdd1.cogroup(rdd2)
```

## PySpark example

In [ ]:
rdd1 = sc.parallelize([("A", 1), ("A", 2), ("B", 3)])
rdd2 = sc.parallelize([("A", 100), ("C", 200)])

result = rdd1.cogroup(rdd2).mapValues(
    lambda x: (list(x[0]), list(x[1]))
)

print(result.collect())

## Expected output

```text
Separate lists from both RDDs for each key
```

## Real-world use case

Advanced reconciliation, multi-value joins, grouped comparison.

## Performance notes

Can consume substantial memory because values are grouped.

## Practice questions

1. Cogroup customers and transactions.
2. Compare grouped source and target values.
3. Explain how cogroup differs from join.

# 30. `subtractByKey()`

## Definition

Removes records from the first Pair RDD when their keys exist in the second Pair RDD.

## Why do we use it?

Use it to exclude keys already processed or already present.

## Transformation classification

- **Type:** Wide
- **Shuffle:** Yes

## Conceptual flow

```text
Remove matching keys, regardless of values
```

## Syntax

```python
rdd1.subtractByKey(rdd2)
```

## PySpark example

In [ ]:
left = sc.parallelize([(1, "A"), (2, "B"), (3, "C")])
right = sc.parallelize([(2, "X")])
print(left.subtractByKey(right).collect())

## Expected output

```text
[(1,'A'), (3,'C')]
```

## Real-world use case

Find new records not yet loaded into the target.

## Performance notes

Requires shuffle.

## Practice questions

1. Find unprocessed customer IDs.
2. Remove blacklisted keys.
3. Compare subtract() and subtractByKey().

# 31. `sortBy()`

## Definition

Sorts an RDD using a custom key function.

## Why do we use it?

Use it when sorting depends on a field or calculated expression.

## Transformation classification

- **Type:** Wide
- **Shuffle:** Yes

## Conceptual flow

```text
Records → custom key → globally sorted RDD
```

## Syntax

```python
rdd.sortBy(keyfunc, ascending=True)
```

## PySpark example

In [ ]:
employees = sc.parallelize([
    ("Anuj", 50000),
    ("Rahul", 70000),
    ("Priya", 60000)
])

print(employees.sortBy(lambda x: x[1], ascending=False).collect())

## Expected output

```text
Employees sorted by descending salary
```

## Real-world use case

Sort by salary, date, score, or record length.

## Performance notes

Global sorting requires shuffle and can be expensive.

## Practice questions

1. Sort strings by length.
2. Sort employees by salary descending.
3. Sort transactions by timestamp.

# 32. `sortByKey()`

## Definition

Sorts a Pair RDD by its key.

## Why do we use it?

Use it when keys must be globally ordered.

## Transformation classification

- **Type:** Wide
- **Shuffle:** Yes

## Conceptual flow

```text
(3,C),(1,A),(2,B) → (1,A),(2,B),(3,C)
```

## Syntax

```python
pair_rdd.sortByKey(ascending=True)
```

## PySpark example

In [ ]:
pairs = sc.parallelize([(3, "C"), (1, "A"), (2, "B")])
print(pairs.sortByKey().collect())

## Expected output

```text
[(1,'A'), (2,'B'), (3,'C')]
```

## Real-world use case

Sort IDs, dates used as keys, or range-partitioned records.

## Performance notes

Requires shuffle and range partitioning.

## Practice questions

1. Sort employee IDs descending.
2. Sort date keys ascending.
3. Compare sortBy and sortByKey.

# 33. `partitionBy()`

## Definition

Applies a partitioning strategy to a Pair RDD.

## Why do we use it?

Use it to place records with the same key in the same partition.

## Transformation classification

- **Type:** Wide
- **Shuffle:** Yes

## Conceptual flow

```text
Hash(key) → target partition
```

## Syntax

```python
pair_rdd.partitionBy(number_of_partitions)
```

## PySpark example

In [ ]:
data = sc.parallelize([
    ("A", 1), ("B", 2), ("A", 3), ("C", 4)
])

partitioned = data.partitionBy(3)

print(partitioned.getNumPartitions())
print(partitioned.glom().collect())

## Expected output

```text
Records distributed across 3 key-based partitions
```

## Real-world use case

Optimize repeated joins and key-based aggregations.

## Performance notes

Initial call shuffles data, but preserved partitioning may reduce future shuffle.

## Practice questions

1. Partition records into 4 partitions.
2. Verify same keys are together.
3. Explain benefits before repeated joins.

# 34. `repartition()`

## Definition

Changes the number of partitions and fully redistributes the data.

## Why do we use it?

Use it to increase parallelism or rebalance uneven partitions.

## Transformation classification

- **Type:** Wide
- **Shuffle:** Yes

## Conceptual flow

```text
Old partitions → full shuffle → new balanced partitions
```

## Syntax

```python
rdd.repartition(number)
```

## PySpark example

In [ ]:
numbers = sc.parallelize(range(1, 21), 2)
print("Before:", numbers.getNumPartitions())

repartitioned = numbers.repartition(5)
print("After:", repartitioned.getNumPartitions())
print(repartitioned.glom().collect())

## Expected output

```text
Partition count changes from 2 to 5
```

## Real-world use case

Increase parallelism before expensive processing or balance skewed partitions.

## Performance notes

Always causes shuffle.

## Practice questions

1. Increase partitions from 2 to 6.
2. Reduce partitions using repartition.
3. Compare distribution before and after.

# 35. `coalesce()`

## Definition

Reduces the number of partitions without a full shuffle by default.

## Why do we use it?

Use it mainly to reduce the number of output files.

## Transformation classification

- **Type:** Narrow by default
- **Shuffle:** Usually No

## Conceptual flow

```text
Several existing partitions merged into fewer partitions
```

## Syntax

```python
rdd.coalesce(number)
```

## PySpark example

In [ ]:
numbers = sc.parallelize(range(1, 21), 5)
print("Before:", numbers.getNumPartitions())

coalesced = numbers.coalesce(2)
print("After:", coalesced.getNumPartitions())
print(coalesced.glom().collect())

## Expected output

```text
Partition count changes from 5 to 2
```

## Real-world use case

Reduce small output files before writing.

## Performance notes

Cheaper than repartition for reducing partitions, but data may become uneven.

## Practice questions

1. Reduce 8 partitions to 2.
2. Compare coalesce and repartition output.
3. Explain when coalesce can create imbalance.

# 36. `repartitionAndSortWithinPartitions()`

## Definition

Repartitions a Pair RDD and sorts records inside each resulting partition.

## Why do we use it?

Use it when sorted records are needed within partitions.

## Transformation classification

- **Type:** Wide
- **Shuffle:** Yes

## Conceptual flow

```text
Shuffle by partitioner → sort inside each partition
```

## Syntax

```python
pair_rdd.repartitionAndSortWithinPartitions(numPartitions=n)
```

## PySpark example

In [ ]:
pairs = sc.parallelize([
    (5, "E"),
    (1, "A"),
    (3, "C"),
    (2, "B"),
    (4, "D")
], 2)

result = pairs.repartitionAndSortWithinPartitions(numPartitions=2)

print(result.glom().collect())

## Expected output

```text
Each partition contains key-sorted records
```

## Real-world use case

Sorted output pipelines, range processing, merge-style workflows.

## Performance notes

Often better than repartition followed by separate sorting.

## Practice questions

1. Use 3 output partitions.
2. Inspect sorted keys inside every partition.
3. Compare with sortByKey.

# Final Comparison: Commonly Confused Transformations

## `map()` vs `flatMap()`

- `map()` returns one output per input.
- `flatMap()` can return multiple outputs and flattens them.

## `groupByKey()` vs `reduceByKey()`

- `groupByKey()` shuffles all values.
- `reduceByKey()` performs local aggregation first.
- Prefer `reduceByKey()` for sums, counts, min, and max.

## `repartition()` vs `coalesce()`

- `repartition()` always shuffles and can increase or decrease partitions.
- `coalesce()` is mainly for reducing partitions and usually avoids full shuffle.

## `join()` vs `cogroup()`

- `join()` creates value pairs.
- `cogroup()` keeps grouped iterables from each RDD separately.

## `sortBy()` vs `sortByKey()`

- `sortBy()` uses a custom key function.
- `sortByKey()` sorts Pair RDD records directly by key.

# End-to-End Practice Project

Use this data:

```python
employee_data = [
    "101,Anuj,IT,80000",
    "102,Rahul,HR,60000",
    "103,Priya,IT,90000",
    "104,Neha,Finance,75000",
    "105,Amit,HR,65000"
]
```

Complete the following:

1. Convert each row into a tuple.
2. Filter employees with salary at least 70000.
3. Create `(department, salary)` Pair RDD.
4. Calculate total salary by department.
5. Calculate employee count by department.
6. Calculate average salary by department.
7. Sort departments by average salary.
8. Repartition the final RDD into 2 partitions.
9. Inspect partition contents using `glom()`.

In [ ]:
employee_data = [
    "101,Anuj,IT,80000",
    "102,Rahul,HR,60000",
    "103,Priya,IT,90000",
    "104,Neha,Finance,75000",
    "105,Amit,HR,65000"
]

raw_rdd = sc.parallelize(employee_data)

parsed = raw_rdd.map(
    lambda row: row.split(",")
).map(
    lambda x: (int(x[0]), x[1], x[2], int(x[3]))
)

high_salary = parsed.filter(lambda x: x[3] >= 70000)

department_salary = parsed.map(lambda x: (x[2], x[3]))

sum_count = department_salary.aggregateByKey((0, 0))(
    lambda acc, salary: (acc[0] + salary, acc[1] + 1),
    lambda left, right: (left[0] + right[0], left[1] + right[1])
)

average_salary = sum_count.mapValues(
    lambda x: x[0] / x[1]
)

sorted_average = average_salary.sortBy(
    lambda x: x[1],
    ascending=False
)

final_rdd = sorted_average.repartition(2)

print("Parsed:", parsed.collect())
print("High salary:", high_salary.collect())
print("Average by department:", average_salary.collect())
print("Sorted:", sorted_average.collect())
print("Partitions:", final_rdd.glom().collect())

# Interview Revision

1. What is lazy evaluation?
2. What is RDD lineage?
3. What is the difference between narrow and wide transformations?
4. Why does shuffle create stage boundaries?
5. Why is `reduceByKey()` preferred over `groupByKey()`?
6. When should `mapPartitions()` be used?
7. What is the difference between `repartition()` and `coalesce()`?
8. How does Spark recover a lost RDD partition?
9. Which transformations preserve partitioning?
10. Why can `cartesian()` be dangerous?

# Stop Spark

In [ ]:
spark.stop()